### load pickle files

In [1]:
import tensorflow as ts
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np


In [2]:
## load all the train models, scalar pickle & onehot 

model = load_model('model.h5')

## load the encoder and scalar

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geo.pkl','rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('scalar.pkl','rb') as file:
    scalar = pickle.load(file)

In [3]:
input_data = {
    'CreditScore':600,	
    'Gender':'Male',
    'Geography':'France',
    'Age':40,	
    'Tenure':3,	
    'Balance':6000,	
    'NumOfProducts':2,	
    'HasCrCard':1,
    'IsActiveMember':1,	
    'EstimatedSalary':50000,
}

In [4]:
## one hot-encoded for Geography

geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]])
geo_encoded_df = pd.DataFrame(geo_encoded.toarray(),columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

/Users/amit/work/AI/ann-clasification/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [5]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Gender,Geography,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,Male,France,40,3,6000,2,1,1,50000


In [6]:
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Gender,Geography,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,1,France,40,3,6000,2,1,1,50000


In [7]:
## combine onehot encoded geograpy to input data
input_df = pd.concat([input_df.drop('Geography',axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,6000,2,1,1,50000,1.0,0.0,0.0


In [8]:
input_scaled = scalar.transform(input_df)
input_scaled

array([[-0.55012981,  0.89091075,  0.12136034, -0.68041201, -1.14498097,
         0.84584804,  0.62670381,  0.968496  , -0.91572441,  1.00400803,
        -0.57427105, -0.58350885]])

In [9]:
## predict churn

predict = model.predict(input_scaled)
predict

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step


array([[1.0564141e-07]], dtype=float32)

In [10]:
prediction_prob = predict[0][0]
prediction_prob
print(f"{prediction_prob:.8f}")

0.00000011


In [11]:
if prediction_prob > 0.5:
  print('customer is likely to churn')
else:
  print('customer is not likely to churn')

customer is not likely to churn
